# Datos masivos
## Clase 2. RDDs
###### Alberto Benavides

## Introducción
- Unidad fundamental de datos en Spark
- Resilient Distributed Dataset
- Resiliente: Tolerante a fallos al reconstruir datos en caso de errores
    - Trabaja por *chunks* de datos
    - Los *chunks* son replicados en diferentes nodos
- Distribuido: Usa nodos (un *cluster* de máquinas) para distribuir datos
- Conjunto de datos: Colección de datos

<center>

![](https://miro.medium.com/v2/resize:fit:828/format:webp/1*eadvOMrshH3HTUq_PmAlYw.png)

</center>

- Datos en los RDDs
    - Inmutables: No pueden ser modificados una vez creados
    - Sujeto a transformaciones: Rápido y fácil acceso a sus datos y transformaciones
    - Paralelo: Los datos se trabajan de manera simultánea

<center>

![](https://miro.medium.com/v2/resize:fit:828/format:webp/1*keJETGFT4dxMYb360TT9IQ.png)
</center>

- Origen de datos
    - Paralelización de datos existentes
    - Referencia a datos externos

* Características técnicas
    * [Muchas clases relacionadas](https://stackoverflow.com/questions/37615149/why-are-there-different-rdds-and-what-are-their-respective-purposes)


In [ ]:
from pyspark import SparkContext

In [ ]:
sc = SparkContext("local", "RDDs")

## Operaciones

### Transformaciones

- Operaciones aplicadas al crear un nuevo RDD
- **Evaluación perezosa (*Lazy Evaluation*)**: Transformaciones no se ejecutan al instante, se almacenan en un "plan de ejecución" que se ejecuta cuando una acción (por ejemplo `collect`) es llamada.

#### Función `map`
- Iteración de elementos de un conjunto de datos sobre una función para formar un nuevo conjunto de datos

In [ ]:
lista = ['b', 'a', 'c']
rdd = sc.parallelize(lista)
rdd_transformado = rdd.map(lambda elemento: elemento.upper())

In [ ]:
# collect devuelve resultados de los rdds
rdd.collect()

['b', 'a', 'c']

In [ ]:
rdd_transformado.collect()

['B', 'A', 'C']

In [ ]:
datos = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(datos)
rdd_transformado = rdd.map(lambda x: x * 2)
resultado = rdd_transformado.collect()
resultado

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]

In [ ]:
type(resultado)

list

#### Función `flatMap`
- Aplica una función a los elementos de un RDD
- Por cada elemento, puede generar de $0$ a $n$ resultados (con base en la función aplicada)
- Regresa un arreglo unidimensional del resultado

<center>

![](https://miro.medium.com/v2/resize:fit:1400/format:webp/0*9u08j6tBBbnXQqHp.gif)

</center>

In [ ]:
import random

In [ ]:
matriz = []
for fila in range(6):
    columnas = random.randint(0, 5)
    fila = []
    for columna in range(columnas):
        fila.append(random.randint(1, 100))
    matriz.append(fila)
matriz
# resultado ejemplo: [[89], [25, 7, 48, 50, 76], [9, 3, 43, 49, 38], [], [42, 98, 51], []]

[[88, 27, 80], [55, 7, 72, 29], [76, 39, 9, 69], [64, 77, 67], [], []]

In [ ]:
rdd = sc.parallelize(matriz)
resultado = rdd.flatMap(lambda x: x)
resultado.collect()

[88, 27, 80, 55, 7, 72, 29, 76, 39, 9, 69, 64, 77, 67]

#### Función `filter`
- Regresa un conjunto de datos con los elementos que cumplen condicióne determinada

<center>

![](https://miro.medium.com/v2/resize:fit:828/format:webp/0*o1FTDAsVL1bQkqdR.gif)

</center>

In [ ]:
x = sc.parallelize([random.randint(1, 100) for x in range(10)])
y = x.filter(lambda x: x % 5 == 0)
print(x.collect())
print(y.collect())

[60, 8, 66, 82, 10, 37, 4, 50, 84, 94]
[60, 10, 50]


#### _Lazy evaluation_
Prepara una transformación para ser ejecutada tras un disparador, como `collect`, de manera que se optimiza el flujo de cómputo.

In [ ]:
datos = [[1, 2], [3, 4, 5], []]
rdd = sc.parallelize(datos)
rdd_aplanado = rdd.flatMap(lambda x: x)
rdd_mapeado = rdd_aplanado.map(lambda x: x * 2)
rdd_filtrado = rdd_mapeado.filter(lambda x: x > 5)
rdd_filtrado

PythonRDD[11] at RDD at PythonRDD.scala:56

In [ ]:
resultado = rdd_filtrado.collect()
resultado

[6, 8, 10]

#### Función `groupByKey`
- Reordena datos a partir de pares llave-valor
- Agrupa los valores a partir de sus llaves $K$
- Genera $K$ grupos iterables como salida

<center>

![](https://miro.medium.com/v2/resize:fit:786/format:webp/1*aucpZLVjS48m6Fgh0vmDXg.png)

</center>

In [ ]:
rdd = sc.parallelize([
    ('fruta', 'melón'),
    ('verdura', 'acelga'),
    ('fruta', 'arándano'),
    ('fruta', 'naranja'),
    ('verdura', 'apio'),
    ('verdura', 'lechuga'),
    ('fruta', 'tomate'),
    ('fruta', 'limón')
])
rdd_agrupado = rdd.groupByKey()
rdd_agrupado.collect()

[('fruta', <pyspark.resultiterable.ResultIterable at 0x79bfbc278ef0>),
 ('verdura', <pyspark.resultiterable.ResultIterable at 0x79bfbc278e90>)]

In [ ]:
rdd_lista = rdd_agrupado.mapValues(list)
resultado = rdd_lista.collect()
resultado

[('fruta', ['melón', 'arándano', 'naranja', 'tomate', 'limón']),
 ('verdura', ['acelga', 'apio', 'lechuga'])]

In [ ]:
resultado[0]

('fruta', ['melón', 'arándano', 'naranja', 'tomate', 'limón'])

- Ejemplo de diccionario con `collectAsMap`

In [ ]:
rdd_lista = rdd_agrupado.mapValues(list)
resultado = rdd_lista.collectAsMap()
resultado

{'fruta': ['melón', 'arándano', 'naranja', 'tomate', 'limón'],
 'verdura': ['acelga', 'apio', 'lechuga']}

In [ ]:
resultado['fruta']

['melón', 'arándano', 'naranja', 'tomate', 'limón']

#### Función `reduceByKey`
- Se usa para aplicar operaciones de agregación en conjuntos de datos
- Suele ser más rápida que `groupByKey`

In [ ]:
rdd = sc.parallelize([
    ("frutas", 2),
    ("frutas", 3),
    ("vegetales", 4),
    ("frutas", 1),
    ("vegetales", 5),
    ("frutas", 7)
])
rdd_sumado = rdd.reduceByKey(lambda x, y: x + y) # Se suma cada par de valores con la misma llave
rdd_sumado.collectAsMap()

{'frutas': 13, 'vegetales': 9}

In [ ]:
rdd_maximo = rdd.reduceByKey(lambda x, y: max(x, y))
rdd_maximo.collectAsMap()

{'frutas': 7, 'vegetales': 5}

#### Funciones `cache` y `persist`
- Almacenan en memoria (`cache`) o disco (`persist`) los resultados de un RDD
- En realidad, `cache` es una función simplificada de `persist`

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5])
rdd_transformado = rdd.map(lambda x: x * 2)
rdd_transformado.cache()

print("Primera acción; se calcula el RDD:", rdd_transformado.collect())
print("Segunda acción; ya con los datos en memoria en cache:", rdd_transformado.collect())

Primera acción; se calcula el RDD: [2, 4, 6, 8, 10]
Segunda acción; ya con los datos en memoria en cache: [2, 4, 6, 8, 10]


- Diferencia en tiempos de ejecución

In [ ]:
import time

In [ ]:
datos = [1, 2, 3, 4, 5] * 10_000_000
rdd = sc.parallelize(datos)
def transformation(x):
    return x * random.randint(1, 5)

# Realmente esto no es tan preciso
# Se require un diseño de experimentos más formal
# Se require un entorno más controlado
inicio = time.time() # Tiempo de procesamiento inicial
sin_cache = rdd.map(transformation).filter(lambda x: x > 10)
sin_cache.collect()
fin = time.time() # Fin de ejecución

print("Tiempo sin cache:", fin - inicio)

Tiempo sin cache: 54.51844024658203


In [ ]:
inicio = time.time() # Tiempo de procesamiento inicial
con_cache = rdd.map(transformation).filter(lambda x: x > 10).cache()
result1 = con_cache.collect()
fin = time.time() # Fin de ejecución

print("Con cache:", fin - inicio)

Con cache: 54.26709318161011


In [ ]:
inicio = time.time()
# Repetir la misma operación
result2 = con_cache.collect()
fin = time.time()

print("Tiempo de ejecución con cache:", fin - inicio)

Tiempo de ejecución con cache: 1.843684434890747


In [ ]:
result1[:10]

[16, 15, 20, 25, 12, 15, 20, 16, 20, 12]

In [ ]:
result2[:10]

[16, 15, 20, 25, 12, 15, 20, 16, 20, 12]

- `persist` usa estos parámetros (que incluyen a `cache`)
    - `MEMORY_ONLY`: Almacena los datos solo en memoria (`cache`)
    - `MEMORY_AND_DISK`: Almacena en memoria, y si no cabe, en disco
    - `MEMORY_ONLY_SER`: Almacena los datos serializados en memoria (útil para ahorrar espacio)
    - `DISK_ONLY`: Almacena los datos únicamente en disco
    - `MEMORY_AND_DISK_SER`: Serializa los datos antes de almacenarlos en memoria y disco

In [ ]:
from pyspark import StorageLevel

In [ ]:
datos = [1, 2, 3, 4, 5]
rdd = sc.parallelize(datos)
rdd_transformado = rdd.map(lambda x: x * 3)

rdd.persist(StorageLevel.MEMORY_ONLY)
rdd_transformado.collect()

[3, 6, 9, 12, 15]

In [ ]:
con_cache.is_cached

True

In [ ]:
# Liberar memoria
con_cache.unpersist()

PythonRDD[35] at RDD at PythonRDD.scala:56

In [ ]:
con_cache.is_cached

False

#### Funciones `coalesce` y `repartition`
- Partición
    - Unidad básica en que se fragmentan los datos
    - Fragmento de un RDD o *DataFrame*
    - Pueden ser procesados de manera independiente por los nodos del contexto de Spark
- Al crear un RDD, automáticamente se le asigna una cantidad de particiones
- `getNumPartitions`: Muestra el número de particiones de un RDD
- `coalesce`: Disminuye el número de particiones
- `repartition`: Aumenta el número de particiones

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5])
rdd.getNumPartitions()

1

In [ ]:
datos = [1, 2, 3, 4, 5] * 10_000_000
rdd_grande = sc.parallelize(datos)
rdd_grande.getNumPartitions()

1

In [ ]:
rdd_reparticionado = rdd.repartition(4)
rdd_reparticionado.getNumPartitions()

4

In [ ]:
rdd_reducido = rdd_reparticionado.coalesce(2)
rdd_reducido.getNumPartitions()

2

- Recomendaciones para particiones (para PC doméstica)
    - Una partición por cada 100 MG a 1 GB de datos (depende del equipo)
    - No exceder la cantidad de memoria disponible asignada a Spark (tomar en cuenta otras aplicaciones en ejecución)
    - No asignar todos los núcleos de CPU como núcleos de Spark
    - Dos o tres particiones por núcleo de Spark
    - Cerrar todas las otras aplicaciones (o definir un servidor dedicado) para este tipo de metodologías


#### Asignación de recursos de Spark

In [ ]:
# Devuelve el número de núcleos de Spark asociado al contexto
sc.defaultParallelism

1



- `set("spark.master", "local[*]")`: Asigna todos los núcleos de CPU a Spark. Cambiar `*` por un entero, asigna esa cantidad exacta de núcleos
- `set("spark.executor.cores", "2")`: Cantidad de núcleos asignados a los ejecutores
    - Ejecutores: Procesos de trabajo que realizan las operaciones en Spark
- `set("spark.executor.memory", "2g")`: Cantidad de memoria alojada para cada ejecutor

In [ ]:
import multiprocessing

multiprocessing.cpu_count()

2

In [ ]:
from pyspark import SparkConf

In [ ]:
# Ejemplo
# conf = SparkConf().setAppName("MiAplicacionModificada")\
#     .set("spark.master", "local[*]")\
#     .set("spark.executor.cores", "2")\
#     .set("spark.executor.memory", "3g")
# sc = SparkContext(conf = conf)

### Acciones de los RDD

- Al ejecutarse, se aplican todas las transformaciones a los RDDs
- Las más comunes
    - `collect`: Colecta todos los elementos (ya lo vimos; también `collectAsMap`)
    - `count`: Regresa el número de elementos
    - `take(n)`: Muestra los primeros $n$ elementos
    - `reduce`: Aplica una función de reducción por pares a todos los elementos y devuelve el resultado
    - `saveAsTextFile`: Guarda el RDD como un archivo de texto

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5])

In [ ]:
rdd.count()

5

In [ ]:
rdd.take(3)

[1, 2, 3]

In [ ]:
rdd.reduce(lambda x, y: x / y)

0.008333333333333333

In [ ]:
rdd.reduce(lambda x, y: min(x, y))

1

In [ ]:
# Guarda a archivo de texto
destino = "/content/guardado.txt"
rdd.saveAsTextFile(destino)

In [ ]:
# Carga desde archivo de texto
cargado = sc.textFile("/content/guardado.txt")
cargado.collect()

['1', '2', '3', '4', '5']

## Ejemplo con Wikipedia

In [ ]:
import requests

In [ ]:
def contenido_Wikipedia(titulo):
    url = f"https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": titulo,
        "prop": "extracts",
        "explaintext": True
    }
    response = requests.get(url, params = params).json()
    pages = response["query"]["pages"]
    return list(pages.values())[0].get("extract", "")

articulos = ["Python (programming language)"]

rdd = sc.parallelize(articulos)
rdd_contenido = rdd.map(lambda titulo: {"title": titulo, "content": contenido_Wikipedia(titulo)})

In [ ]:
resultado = rdd_contenido.collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 41.0 failed 1 times, most recent failure: Lost task 0.0 in stage 41.0 (TID 39) (cec6d2eda706 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 976, in json
    return complexjson.loads(self.text, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/simplejson/__init__.py", line 514, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/simplejson/decoder.py", line 386, in decode
    obj, end = self.raw_decode(s)
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/simplejson/decoder.py", line 416, in raw_decode
    retur

In [ ]:
for article in resultado:
    print(f"Título: {article['title']} \n Contenido: {article['content'][:200]}...\n")

TypeError: string indices must be integers, not 'str'

## Ejemplo con iris-data

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data" # https://archive.ics.uci.edu/dataset/53/iris
data = requests.get(url).text
data

'5.1,3.5,1.4,0.2,Iris-setosa\n4.9,3.0,1.4,0.2,Iris-setosa\n4.7,3.2,1.3,0.2,Iris-setosa\n4.6,3.1,1.5,0.2,Iris-setosa\n5.0,3.6,1.4,0.2,Iris-setosa\n5.4,3.9,1.7,0.4,Iris-setosa\n4.6,3.4,1.4,0.3,Iris-setosa\n5.0,3.4,1.5,0.2,Iris-setosa\n4.4,2.9,1.4,0.2,Iris-setosa\n4.9,3.1,1.5,0.1,Iris-setosa\n5.4,3.7,1.5,0.2,Iris-setosa\n4.8,3.4,1.6,0.2,Iris-setosa\n4.8,3.0,1.4,0.1,Iris-setosa\n4.3,3.0,1.1,0.1,Iris-setosa\n5.8,4.0,1.2,0.2,Iris-setosa\n5.7,4.4,1.5,0.4,Iris-setosa\n5.4,3.9,1.3,0.4,Iris-setosa\n5.1,3.5,1.4,0.3,Iris-setosa\n5.7,3.8,1.7,0.3,Iris-setosa\n5.1,3.8,1.5,0.3,Iris-setosa\n5.4,3.4,1.7,0.2,Iris-setosa\n5.1,3.7,1.5,0.4,Iris-setosa\n4.6,3.6,1.0,0.2,Iris-setosa\n5.1,3.3,1.7,0.5,Iris-setosa\n4.8,3.4,1.9,0.2,Iris-setosa\n5.0,3.0,1.6,0.2,Iris-setosa\n5.0,3.4,1.6,0.4,Iris-setosa\n5.2,3.5,1.5,0.2,Iris-setosa\n5.2,3.4,1.4,0.2,Iris-setosa\n4.7,3.2,1.6,0.2,Iris-setosa\n4.8,3.1,1.6,0.2,Iris-setosa\n5.4,3.4,1.5,0.4,Iris-setosa\n5.2,4.1,1.5,0.1,Iris-setosa\n5.5,4.2,1.4,0.2,Iris-setosa\n4.9,3.1,1.5,0

In [ ]:
data.strip().split("\n")

['5.1,3.5,1.4,0.2,Iris-setosa',
 '4.9,3.0,1.4,0.2,Iris-setosa',
 '4.7,3.2,1.3,0.2,Iris-setosa',
 '4.6,3.1,1.5,0.2,Iris-setosa',
 '5.0,3.6,1.4,0.2,Iris-setosa',
 '5.4,3.9,1.7,0.4,Iris-setosa',
 '4.6,3.4,1.4,0.3,Iris-setosa',
 '5.0,3.4,1.5,0.2,Iris-setosa',
 '4.4,2.9,1.4,0.2,Iris-setosa',
 '4.9,3.1,1.5,0.1,Iris-setosa',
 '5.4,3.7,1.5,0.2,Iris-setosa',
 '4.8,3.4,1.6,0.2,Iris-setosa',
 '4.8,3.0,1.4,0.1,Iris-setosa',
 '4.3,3.0,1.1,0.1,Iris-setosa',
 '5.8,4.0,1.2,0.2,Iris-setosa',
 '5.7,4.4,1.5,0.4,Iris-setosa',
 '5.4,3.9,1.3,0.4,Iris-setosa',
 '5.1,3.5,1.4,0.3,Iris-setosa',
 '5.7,3.8,1.7,0.3,Iris-setosa',
 '5.1,3.8,1.5,0.3,Iris-setosa',
 '5.4,3.4,1.7,0.2,Iris-setosa',
 '5.1,3.7,1.5,0.4,Iris-setosa',
 '4.6,3.6,1.0,0.2,Iris-setosa',
 '5.1,3.3,1.7,0.5,Iris-setosa',
 '4.8,3.4,1.9,0.2,Iris-setosa',
 '5.0,3.0,1.6,0.2,Iris-setosa',
 '5.0,3.4,1.6,0.4,Iris-setosa',
 '5.2,3.5,1.5,0.2,Iris-setosa',
 '5.2,3.4,1.4,0.2,Iris-setosa',
 '4.7,3.2,1.6,0.2,Iris-setosa',
 '4.8,3.1,1.6,0.2,Iris-setosa',
 '5.4,3.

In [ ]:
rdd = sc.parallelize(data.strip().split("\n"))
rdd.collect()

['5.1,3.5,1.4,0.2,Iris-setosa',
 '4.9,3.0,1.4,0.2,Iris-setosa',
 '4.7,3.2,1.3,0.2,Iris-setosa',
 '4.6,3.1,1.5,0.2,Iris-setosa',
 '5.0,3.6,1.4,0.2,Iris-setosa',
 '5.4,3.9,1.7,0.4,Iris-setosa',
 '4.6,3.4,1.4,0.3,Iris-setosa',
 '5.0,3.4,1.5,0.2,Iris-setosa',
 '4.4,2.9,1.4,0.2,Iris-setosa',
 '4.9,3.1,1.5,0.1,Iris-setosa',
 '5.4,3.7,1.5,0.2,Iris-setosa',
 '4.8,3.4,1.6,0.2,Iris-setosa',
 '4.8,3.0,1.4,0.1,Iris-setosa',
 '4.3,3.0,1.1,0.1,Iris-setosa',
 '5.8,4.0,1.2,0.2,Iris-setosa',
 '5.7,4.4,1.5,0.4,Iris-setosa',
 '5.4,3.9,1.3,0.4,Iris-setosa',
 '5.1,3.5,1.4,0.3,Iris-setosa',
 '5.7,3.8,1.7,0.3,Iris-setosa',
 '5.1,3.8,1.5,0.3,Iris-setosa',
 '5.4,3.4,1.7,0.2,Iris-setosa',
 '5.1,3.7,1.5,0.4,Iris-setosa',
 '4.6,3.6,1.0,0.2,Iris-setosa',
 '5.1,3.3,1.7,0.5,Iris-setosa',
 '4.8,3.4,1.9,0.2,Iris-setosa',
 '5.0,3.0,1.6,0.2,Iris-setosa',
 '5.0,3.4,1.6,0.4,Iris-setosa',
 '5.2,3.5,1.5,0.2,Iris-setosa',
 '5.2,3.4,1.4,0.2,Iris-setosa',
 '4.7,3.2,1.6,0.2,Iris-setosa',
 '4.8,3.1,1.6,0.2,Iris-setosa',
 '5.4,3.

In [ ]:
rdd_num = rdd.map(lambda linea: linea.split(",")[:4]) \
             .map(lambda valores: list(map(float, valores)))
rdd_num.collect()

[[5.1, 3.5, 1.4, 0.2],
 [4.9, 3.0, 1.4, 0.2],
 [4.7, 3.2, 1.3, 0.2],
 [4.6, 3.1, 1.5, 0.2],
 [5.0, 3.6, 1.4, 0.2],
 [5.4, 3.9, 1.7, 0.4],
 [4.6, 3.4, 1.4, 0.3],
 [5.0, 3.4, 1.5, 0.2],
 [4.4, 2.9, 1.4, 0.2],
 [4.9, 3.1, 1.5, 0.1],
 [5.4, 3.7, 1.5, 0.2],
 [4.8, 3.4, 1.6, 0.2],
 [4.8, 3.0, 1.4, 0.1],
 [4.3, 3.0, 1.1, 0.1],
 [5.8, 4.0, 1.2, 0.2],
 [5.7, 4.4, 1.5, 0.4],
 [5.4, 3.9, 1.3, 0.4],
 [5.1, 3.5, 1.4, 0.3],
 [5.7, 3.8, 1.7, 0.3],
 [5.1, 3.8, 1.5, 0.3],
 [5.4, 3.4, 1.7, 0.2],
 [5.1, 3.7, 1.5, 0.4],
 [4.6, 3.6, 1.0, 0.2],
 [5.1, 3.3, 1.7, 0.5],
 [4.8, 3.4, 1.9, 0.2],
 [5.0, 3.0, 1.6, 0.2],
 [5.0, 3.4, 1.6, 0.4],
 [5.2, 3.5, 1.5, 0.2],
 [5.2, 3.4, 1.4, 0.2],
 [4.7, 3.2, 1.6, 0.2],
 [4.8, 3.1, 1.6, 0.2],
 [5.4, 3.4, 1.5, 0.4],
 [5.2, 4.1, 1.5, 0.1],
 [5.5, 4.2, 1.4, 0.2],
 [4.9, 3.1, 1.5, 0.1],
 [5.0, 3.2, 1.2, 0.2],
 [5.5, 3.5, 1.3, 0.2],
 [4.9, 3.1, 1.5, 0.1],
 [4.4, 3.0, 1.3, 0.2],
 [5.1, 3.4, 1.5, 0.2],
 [5.0, 3.5, 1.3, 0.3],
 [4.5, 2.3, 1.3, 0.3],
 [4.4, 3.2, 1.3, 0.2],
 [5.0, 3.5,

In [ ]:
petal_len = rdd_num.map(lambda x: x[2])

print("Conteo:", petal_len.count())
print("Media:", petal_len.mean())
print("Desv. estándar:", petal_len.stdev())
print("Mínimo:", petal_len.min())
print("Máximo:", petal_len.max())

Conteo: 150
Media: 3.758666666666667
Desv. estándar: 1.7585291834055203
Mínimo: 1.0
Máximo: 6.9


In [ ]:
# A partir de reduce
n = petal_len.count()
suma = petal_len.reduce(lambda a, b: a + b)
media = suma / n
desv_standar = (petal_len.map(lambda x: (x - media)**2).reduce(lambda a, b: a + b) / n) ** (1/2)

print("Media:", media)
print("Desv. estándar:", desv_standar)

Media: 3.7586666666666693
Desv. estándar: 1.7585291834055201


## Tarea 2 (10 puntos). Operaciones con RDDs
- Obtener información de algún origen de datos (propio o de API)
- Convertir el origen de datos a RDD con pySpark
- Realizar alguna operación en el RDD, como estadísticas descriptivas básicas
- Subir la práctica a un repositorio público y etiquetarla claramente

## Referencias
- Learning Spark: Lightning-Fast Big Data Analysis - Karau, Holden, et al. (O'Reilly, 2015).
- Spark: The Definitive Guide - Bill Chambers, Matei Zaharia (O'Reilly, 2018).
- https://spark.apache.org/docs/latest/api/python/
- https://medium.com/@vaishalisubbaraj/lets-learn-about-rdd-5bf65c8c2477
- https://medium.com/@john_tringham/spark-concepts-simplified-lazy-evaluation-d398891e0568